# M06 — See the Data Before Modeling It

**Objective:** use visualization to interrogate data before choosing or fitting a model.

Guided decision question:

> Which observable signals are associated with escalation in this support-ticket sample, and what data-quality or leakage risks must be resolved before any escalation model is attempted?

This is a local, deterministic, CPU-only lab. It does not fit a model, call a network service, use a secret, or pre-populate learner evidence.

## Start with questions, not chart types

For every section use this loop:

**question → prediction → chart rationale → run → visible observation → inference → limitation → next question**

**Predict before running:** Which two fields do you expect to be most associated with escalation? Which field is most likely to be unavailable when a ticket first opens? Record the prediction outside the source notebook before continuing.

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

get_ipython().run_line_magic("matplotlib", "inline")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "missions.json").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the LearningOS-AI repository.")

ROOT = find_repo_root()
GUIDED_PATH = ROOT / "datasets" / "M06" / "support_tickets.csv"
FRESH_PATH = ROOT / "datasets" / "M06" / "community_programs_fresh.csv"
print("Repository root:", ROOT)
print("Guided fixture: ", GUIDED_PATH.relative_to(ROOT))

## 1. Establish row grain and field timing

Question: what does one row represent, and when does each field become available?

**Predict before running:** Which validation would fail first if the file had duplicate ticket IDs or an unexpected target value? Why must field timing be understood before looking for predictive patterns?

In [ ]:
tickets = pd.read_csv(GUIDED_PATH)
required_columns = {
    "ticket_id", "opened_week", "channel", "region", "issue_type",
    "customer_tenure_months", "first_response_minutes", "messages_count",
    "satisfaction_score", "escalated", "post_case_priority",
}
assert required_columns == set(tickets.columns)
assert tickets["ticket_id"].is_unique
assert set(tickets["escalated"].unique()) == {0, 1}
assert len(tickets) == 48
print(f"Row grain: one closed ticket | rows={len(tickets)} | columns={tickets.shape[1]}")
display(tickets.head(8))

In [ ]:
field_timing = pd.DataFrame(
    [
        ("channel", "ticket opening", True),
        ("region", "ticket opening", True),
        ("issue_type", "initial categorization", True),
        ("customer_tenure_months", "ticket opening", True),
        ("first_response_minutes", "after first response", False),
        ("messages_count", "case closure", False),
        ("satisfaction_score", "optional post-case survey", False),
        ("post_case_priority", "case closure", False),
    ],
    columns=["field", "first_available", "available_when_ticket_opens"],
)
display(field_timing)

### Written reasoning contract

After every view, keep these labels separate:

- **Visible observation:** a statement traceable to a mark, count, table value, or field definition.
- **Inference:** a plausible explanation or operational hypothesis.
- **What cannot yet be concluded:** causal, population, or future-performance claims not established by this fixture.

The worked statements below model this separation. They are not learner evidence; write your own record before comparing.

## 2. Distribution

Question: what is the shape and typical range of first-response time?

**Predict before running:** Will the distribution look symmetric, left-tailed, or right-tailed? Name the evidence that would change your prediction.

Chart rationale: a histogram answers a shape question by showing concentration and tails while retaining the measurement scale.

In [ ]:
distribution_stats = tickets["first_response_minutes"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
).round(2)
display(distribution_stats.to_frame("first_response_minutes"))

In [ ]:
response_bins = [0, 10, 20, 40, 60, 120, 240, 500]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(
    tickets["first_response_minutes"],
    bins=response_bins,
    color="#35618d",
    edgecolor="white",
)
ax.set(
    title="How are first-response times distributed?",
    xlabel="First-response time (minutes)",
    ylabel="Ticket count",
)
plt.show()
plt.close(fig)

**Visible observation:** most response times occupy the lower bins, while a small number extend far to the right. The median and maximum are far apart.

**Inference:** the process may contain a common fast-response regime and a smaller delayed regime worth tracing to channel, workload, or issue type.

**What cannot yet be concluded:** the chart does not establish why delays occurred, whether they are errors, or whether this synthetic sample represents another operation. Bin boundaries also hide within-bin variation.

## 3. Missingness

Question: which fields are missing, how much is missing, and could absence itself be structured?

**Predict before running:** Which post-case field is most likely to be missing? Why is replacing every blank with zero unsafe?

Chart rationale: a missing-count bar chart compares completeness across fields without pretending blanks are measured zeros.

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_count": tickets.isna().sum(),
        "missing_rate": tickets.isna().mean(),
    }
).sort_values(["missing_count", "missing_rate"], ascending=False)
display(missing_summary)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(missing_summary.index, missing_summary["missing_count"], color="#8e6c4a")
ax.set(title="Which fields are incomplete?", ylabel="Missing records")
ax.tick_params(axis="x", rotation=55)
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
survey_missing_by_target = (
    tickets.groupby("escalated")["satisfaction_score"]
    .apply(lambda values: values.isna().mean())
    .rename("satisfaction_missing_rate")
)
display(survey_missing_by_target.to_frame())

**Visible observation:** missing values occur in customer tenure and the optional satisfaction survey; the missing survey share differs between the two outcome groups in this small sample.

**Inference:** survey absence may reflect the collection process rather than random loss, so complete-case analysis could select a different mix of tickets.

**What cannot yet be concluded:** group differences in missingness do not reveal why a person skipped a survey. The fixture does not support a claim that missingness would behave the same in production.

## 4. Outliers

Question: which response-time values are unusually far from the central mass, and should they be investigated rather than deleted?

**Predict before running:** How many tickets do you expect the IQR rule to flag? What evidence would distinguish a data error from a rare operational delay?

In [ ]:
def iqr_outliers(frame, column):
    values = frame[column].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    mask = frame[column].lt(lower_fence) | frame[column].gt(upper_fence)
    return (lower_fence, upper_fence), frame.loc[mask].copy()

response_fences, response_outliers = iqr_outliers(
    tickets, "first_response_minutes"
)
print("IQR fences:", tuple(round(value, 2) for value in response_fences))
display(
    response_outliers[
        ["ticket_id", "channel", "issue_type", "first_response_minutes", "escalated"]
    ].sort_values("first_response_minutes")
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.boxplot(
    tickets["first_response_minutes"].dropna(),
    vert=False,
    patch_artist=True,
    boxprops={"facecolor": "#86a6c3"},
)
ax.set(
    title="Which response times deserve row-level investigation?",
    xlabel="First-response time (minutes)",
)
ax.set_yticks([])
plt.show()
plt.close(fig)

**Visible observation:** the IQR rule identifies multiple high response-time rows beyond the upper fence; the boxplot shows them separated from the central box.

**Inference:** these rows may represent real severe delays, a distinct workflow, or data-entry problems. Their ticket IDs make follow-up possible.

**What cannot yet be concluded:** an IQR flag does not prove a record is wrong and is not a deletion rule. Removing these rows without operational evidence would erase the cases the escalation question may care about most.

## 5. Relationships

Question: how do first-response time and satisfaction move together among tickets with an observed survey response?

**Predict before running:** Which direction do you expect, and where might escalated tickets appear? State why a relationship would still not prove causation.

Chart rationale: a scatter plot preserves each observed pair and can reveal overlap, clusters, and extreme rows hidden by averages.

In [ ]:
paired = tickets.dropna(subset=["first_response_minutes", "satisfaction_score"])
fig, ax = plt.subplots(figsize=(9, 5))
for target, label, color in [
    (0, "not escalated", "#4e7a63"),
    (1, "escalated", "#b5533c"),
]:
    subset = paired.loc[paired["escalated"].eq(target)]
    ax.scatter(
        subset["first_response_minutes"],
        subset["satisfaction_score"],
        label=f"{label} (n={len(subset)})",
        color=color,
        alpha=0.8,
        s=55,
    )
ax.set(
    title="How do response time and observed satisfaction vary together?",
    xlabel="First-response time (minutes)",
    ylabel="Satisfaction score (1–5)",
)
ax.set_ylim(0.5, 5.5)
ax.legend()
plt.show()
plt.close(fig)
display(paired[["first_response_minutes", "satisfaction_score"]].corr())

**Visible observation:** lower satisfaction values occur more often at longer response times in the observed pairs, and escalation markers are concentrated toward longer waits and lower scores. The groups still overlap.

**Inference:** response delay could be a useful diagnostic signal, while issue severity or channel could influence both wait and outcome.

**What cannot yet be concluded:** the display cannot show that delay caused low satisfaction or escalation. Satisfaction is post-case and selectively missing, so it is not an opening-time feature and the plotted subset may be biased.

## 6. Groups

Question: do escalation rates differ by channel, region, or issue type, and are the denominators comparable?

**Predict before running:** Which channel will have the highest rate? Record both the expected rate ordering and the group-size caveat.

Chart rationale: rate bars answer a group comparison only when counts are retained alongside the rates.

In [ ]:
def escalation_summary(frame, group_column):
    summary = (
        frame.groupby(group_column, as_index=False)
        .agg(
            ticket_count=("ticket_id", "size"),
            escalated_count=("escalated", "sum"),
            escalation_rate=("escalated", "mean"),
        )
        .sort_values("escalation_rate", ascending=False)
    )
    return summary

channel_summary = escalation_summary(tickets, "channel")
region_summary = escalation_summary(tickets, "region")
issue_summary = escalation_summary(tickets, "issue_type")
display(channel_summary)
display(region_summary)
display(issue_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    channel_summary["channel"],
    channel_summary["escalation_rate"],
    color="#567a9f",
)
for bar, count in zip(bars, channel_summary["ticket_count"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f"n={count}",
        ha="center",
    )
ax.set(
    title="How does escalation rate vary by channel?",
    xlabel="Channel",
    ylabel="Escalation rate",
)
ax.set_ylim(0, max(0.5, channel_summary["escalation_rate"].max() + 0.1))
plt.show()
plt.close(fig)

**Visible observation:** escalation rates differ across channel, region, and issue-type groups in this fixture; each table shows the numerator and denominator behind its rate.

**Inference:** channel or workflow differences are candidates for follow-up, but apparent group effects may reflect different issue mixes or response delays within groups.

**What cannot yet be concluded:** group ordering in 48 designed records is not a stable ranking for future tickets. Aggregates do not isolate competing explanations or quantify uncertainty.

## 7. Class imbalance

Question: is the escalation target imbalanced enough that raw accuracy would be a weak future baseline?

**Predict before running:** What accuracy would a rule that always predicts the majority class achieve? Why could that still be useless?

In [ ]:
class_counts = tickets["escalated"].value_counts().sort_index()
class_summary = pd.DataFrame(
    {
        "ticket_count": class_counts,
        "class_share": class_counts / class_counts.sum(),
    }
)
class_summary.index = ["not escalated", "escalated"]
majority_baseline = class_counts.max() / class_counts.sum()
display(class_summary)
print(f"Majority-class accuracy baseline: {majority_baseline:.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(class_summary.index, class_summary["ticket_count"], color=["#547a67", "#b5533c"])
ax.set(title="How imbalanced is the escalation outcome?", ylabel="Ticket count")
plt.show()
plt.close(fig)

**Visible observation:** 10 of 48 tickets are escalated, so the majority class contains 38 tickets and a majority-only rule would report about 79% accuracy.

**Inference:** a later modeling mission would need metrics that expose minority-class misses rather than relying on accuracy alone.

**What cannot yet be concluded:** this class share is designed for the exercise. It does not estimate a real escalation prevalence or determine which metric has the correct business cost.

## 8. Possible leakage

Question: does any field contain information that would not exist when a new ticket is scored?

**Predict before running:** Which field will show the strongest association with escalation? Use field timing—not only correlation—to decide whether it is a legitimate candidate feature.

In [ ]:
leakage_counts = pd.crosstab(
    tickets["escalated"],
    tickets["post_case_priority"],
    margins=True,
)
leakage_rates = pd.crosstab(
    tickets["escalated"],
    tickets["post_case_priority"],
    normalize="index",
)
display(leakage_counts)
display(leakage_rates)
assert tickets.loc[tickets["escalated"].eq(1), "post_case_priority"].eq("urgent").all()
assert tickets.loc[tickets["escalated"].eq(0), "post_case_priority"].eq("standard").all()

**Visible observation:** `post_case_priority` perfectly separates the target in this fixture, and the field map says it is assigned only at case closure.

**Inference:** the field is a deliberate leakage trap: it encodes knowledge of the completed outcome and would create unrealistic evaluation if used to predict escalation at opening.

**What cannot yet be concluded:** a strong association alone does not always prove leakage. The decisive evidence here is the operational timing definition. Other post-opening fields require a prediction-moment decision before inclusion or exclusion.

## 9. Controlled failure — a misleading visualization

Question: can an axis choice make modest group differences look operationally dramatic?

**Predict before running:** Which chart will exaggerate the apparent channel difference? What must remain identical for this to isolate the effect of a truncated scale?

In [ ]:
channel_tenure = (
    tickets.groupby("channel")["customer_tenure_months"]
    .agg(["mean", "count"])
    .sort_values("mean")
)
display(channel_tenure)

plot_values = channel_tenure["mean"]
truncated_low = max(0, float(plot_values.min()) - 0.5)
truncated_high = float(plot_values.max()) + 0.5
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].bar(channel_tenure.index, plot_values, color="#b5533c")
axes[0].set_title("Misleading: truncated scale")
axes[0].set_ylim(truncated_low, truncated_high)
axes[0].set_ylabel("Mean customer tenure (months)")

axes[1].bar(channel_tenure.index, plot_values, color="#4e7a63")
axes[1].set_title("Context: zero baseline and sample range")
axes[1].set_ylim(0, 50)
axes[1].set_ylabel("Mean customer tenure (months)")

for axis in axes:
    axis.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()
plt.close(fig)

**Visible observation:** both panels use the same means and ordering, but the truncated scale makes the bars appear much farther apart.

**Inference:** a reader could overstate the operational importance of modest mean differences if the scale and denominators are not inspected.

**What cannot yet be concluded:** neither chart establishes a channel effect, statistical stability, or a scheduling decision. Group means also hide within-channel distributions.

Repair checklist: show an honest scale for bars, retain exact values and counts, consider a record-level distribution view, and document any axis restriction explicitly.

## 10. Code reading before charting

Choose one plotting cell and trace source rows, filters, selected columns, transformations, aggregation, denominators, axes, scale, bins, colour, and lost information.

Then inspect `iqr_outliers` and `escalation_summary`. Explain why each returns auditable intermediate values instead of only a finished chart. Use `missions/M06/code_reading.md` for the complete trace.

In [ ]:
interrogation_audit = {
    "row_grain_checked": tickets["ticket_id"].is_unique,
    "missing_fields": missing_summary.query("missing_count > 0").index.tolist(),
    "iqr_outlier_count": len(response_outliers),
    "group_denominators_retained": "ticket_count" in channel_summary.columns,
    "minority_class_share": float(class_counts.min() / class_counts.sum()),
    "leakage_candidate": "post_case_priority",
    "model_fitted": False,
}
display(pd.Series(interrogation_audit, name="M06 audit"))

## 11. No-AI Gate — fresh dataset and question

Complete `missions/M06/no_ai_gate.md` without AI assistance. The fresh question asks whether weekend community programs appear to have a different no-show rate from weekday programs and what must be investigated before changing the schedule.

You must choose the chart, interpret it, retain denominators, and state limitations. The next cell validates only the fresh file's contract; it deliberately does not calculate the answer or choose a chart.

In [ ]:
fresh = pd.read_csv(FRESH_PATH)
fresh_required = {
    "program_id", "day_type", "format", "topic",
    "registered", "attended", "promotion_channel",
}
assert fresh_required == set(fresh.columns)
assert fresh["program_id"].is_unique
assert set(fresh["day_type"]) == {"weekday", "weekend"}
assert fresh["attended"].le(fresh["registered"]).all()
print(f"Fresh fixture ready: rows={len(fresh)}, columns={fresh.shape[1]}")
print("Column names:", fresh.columns.tolist())

## 12. Assessment and evidence

Submit the evidence defined in `missions/M06/evidence_contract.yaml`. Passing M06 means you can:

- start with a question and prediction rather than a preferred chart type;
- interrogate distribution, missingness, outliers, relationships, groups, imbalance, and field timing;
- preserve denominators and trace chart inputs;
- separate **Visible observation**, **Inference**, and **What cannot yet be concluded**;
- diagnose a misleading but executable visualization;
- transfer the method to a fresh dataset without AI-generated analysis.

Repository implementation status is not learner completion.